In [1]:
import os

import joblib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from torch.utils.data import DataLoader, TensorDataset
from tqdm.auto import tqdm


# ============================================================
# File paths and prediction settings
# ============================================================
INPUT_DATA_PATH = "data/new_data_features_500.csv"

SCALER_PATH = "model/scaler_schemeD_pretrained.pkl"

FINETUNED_MODEL_PATH = (
    "model/mlp_schemeD_final_finetuned.pth"
)

OUTPUT_DIRECTORY = "results"

OUTPUT_FILE_PATH = os.path.join(
    OUTPUT_DIRECTORY,
    "five_metal_compounds_predictions_500.csv",
)

BATCH_SIZE = 512

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

os.makedirs(
    OUTPUT_DIRECTORY,
    exist_ok=True,
)

print("Device:", DEVICE)

Device: cpu


In [2]:
class MLPRegressor(nn.Module):
    """Multilayer perceptron used for overpotential prediction."""

    def __init__(
        self,
        input_dim,
        hidden_dims=None,
        dropout=0.02,
    ):
        super().__init__()

        if hidden_dims is None:
            hidden_dims = [256, 128, 64]

        feature_layers = []
        layer_dims = [input_dim] + hidden_dims

        for index in range(len(hidden_dims)):
            feature_layers.extend(
                [
                    nn.Linear(
                        layer_dims[index],
                        layer_dims[index + 1],
                    ),
                    nn.BatchNorm1d(
                        layer_dims[index + 1]
                    ),
                    nn.ReLU(),
                    nn.Dropout(dropout),
                ]
            )

        self.feature_extractor = nn.Sequential(
            *feature_layers
        )

        self.regressor = nn.Linear(
            hidden_dims[-1],
            1,
        )

    def forward(self, inputs):
        features = self.feature_extractor(inputs)
        return self.regressor(features)

In [3]:
# ============================================================
# Load the candidate feature table
# ============================================================
candidate_df = pd.read_csv(
    INPUT_DATA_PATH
)

excluded_columns = [
    column
    for column in [
        "compound",
        "composition_key",
        "overpotential",
    ]
    if column in candidate_df.columns
]

candidate_feature_df = candidate_df.drop(
    columns=excluded_columns
)

candidate_features = (
    candidate_feature_df
    .to_numpy()
    .astype(np.float32)
)

feature_names = (
    candidate_feature_df.columns.tolist()
)

print(
    "Candidate table shape:",
    candidate_df.shape,
)

print(
    "Feature matrix shape:",
    candidate_features.shape,
)

print(
    "Number of input features:",
    len(feature_names),
)

Candidate table shape: (415800, 27)
Feature matrix shape: (415800, 25)
Number of input features: 25


In [4]:
# ============================================================
# Apply the scaler fitted on the source training set
# ============================================================
scaler = joblib.load(
    SCALER_PATH
)

expected_feature_count = getattr(
    scaler,
    "n_features_in_",
    candidate_features.shape[1],
)

if candidate_features.shape[1] != expected_feature_count:
    raise ValueError(
        "The number of candidate features does not match "
        f"the scaler input dimension. Expected "
        f"{expected_feature_count}, but received "
        f"{candidate_features.shape[1]}."
    )

candidate_features_scaled = scaler.transform(
    candidate_features
).astype(np.float32)

print(
    "Scaled feature matrix shape:",
    candidate_features_scaled.shape,
)

Scaled feature matrix shape: (415800, 25)


In [5]:
# ============================================================
# Load the fine-tuned MLP model
# ============================================================
model = MLPRegressor(
    input_dim=candidate_features_scaled.shape[1],
    hidden_dims=[256, 128, 64],
    dropout=0.02,
).to(DEVICE)

model_state = torch.load(
    FINETUNED_MODEL_PATH,
    map_location=DEVICE,
)

model.load_state_dict(
    model_state
)

model.eval()

print(
    "Fine-tuned model loaded from:",
    FINETUNED_MODEL_PATH,
)

Fine-tuned model loaded from: model/mlp_schemeD_final_finetuned.pth


In [6]:
# ============================================================
# Construct the prediction data loader
# ============================================================
candidate_dataset = TensorDataset(
    torch.from_numpy(
        candidate_features_scaled
    )
)

candidate_loader = DataLoader(
    candidate_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
)


# ============================================================
# Generate predictions
# ============================================================
predicted_overpotentials = []

model.eval()

with torch.no_grad():
    for batch_features, in tqdm(
        candidate_loader,
        desc="Predicting overpotentials",
    ):
        batch_features = batch_features.to(
            DEVICE
        )

        batch_predictions = (
            model(batch_features)
            .cpu()
            .numpy()
            .ravel()
        )

        predicted_overpotentials.extend(
            batch_predictions
        )

predicted_overpotentials = np.asarray(
    predicted_overpotentials
)

if len(predicted_overpotentials) != len(candidate_df):
    raise ValueError(
        "The number of predictions does not match "
        "the number of candidate compositions."
    )

print(
    "Number of predictions:",
    len(predicted_overpotentials),
)

print(
    "Prediction range:",
    predicted_overpotentials.min(),
    "to",
    predicted_overpotentials.max(),
)

Predicting overpotentials:   0%|          | 0/813 [00:00<?, ?it/s]

Number of predictions: 415800
Prediction range: 0.74268144 to 2.7513146


In [7]:
# ============================================================
# Save candidate compositions and predicted overpotentials
# ============================================================
prediction_df = candidate_df.copy()

prediction_df[
    "Predicted_Overpotential"
] = predicted_overpotentials

prediction_df.to_csv(
    OUTPUT_FILE_PATH,
    index=False,
    encoding="utf-8-sig",
)

print(
    "Prediction results saved to:",
    OUTPUT_FILE_PATH,
)

print(
    "Output shape:",
    prediction_df.shape,
)

display(
    prediction_df.head()
)

Prediction results saved to: results\five_metal_compounds_predictions_500.csv
Output shape: (415800, 28)


,compound,metal_x_max,metal1_AN,metal2_AN,metal3_AN,metal4_AN,metal5_AN,M_AN,M_AN_mismatch,M_AR,...,M_Nd_mismatch,M_Nv,M_Nv_mismatch,C_EN,N_metal,s_mix_metal,anion_to_metal_ratio,CurrentDensity,composition_key,Predicted_Overpotential
0,Mn0.2Fe0.1Co0.1Ni0.1Ti0.5N,0.5,22,25,26,27,28,24.10,1.2100,1.6590,...,1.2100,6.10,1.2100,3.04,5,1.359237,1.0,500,Mn0.2Fe0.1Co0.1Ni0.1Ti0.5N,1.837824
1,Mn0.2Fe0.15Co0.1Ni0.05Ti0.5N,0.5,22,25,26,27,28,24.00,1.1350,1.6625,...,1.1350,6.00,1.1350,3.04,5,1.333074,1.0,500,Mn0.2Fe0.15Co0.1Ni0.05Ti0.5N,1.836769
2,Mn0.2Fe0.2Co0.05Ni0.05Ti0.5N,0.5,22,25,26,27,28,23.95,1.0975,1.6645,...,1.0975,5.95,1.0975,3.04,5,1.289922,1.0,500,Mn0.2Fe0.2Co0.05Ni0.05Ti0.5N,1.848467
3,Mn0.2Fe0.1Co0.1Cu0.1Ti0.5N,0.5,22,25,26,27,29,24.20,1.3000,1.6550,...,1.3900,6.20,1.3000,3.04,5,1.359237,1.0,500,Mn0.2Fe0.1Co0.1Cu0.1Ti0.5N,1.791909
4,Mn0.2Fe0.15Co0.1Cu0.05Ti0.5N,0.5,22,25,26,27,29,24.05,1.1825,1.6605,...,1.2300,6.05,1.1825,3.04,5,1.333074,1.0,500,Mn0.2Fe0.15Co0.1Cu0.05Ti0.5N,1.827055
